In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio

import yaml

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

--------
# Sanity check: compare the no-magnification counts with original catalog

In [10]:
cat = Table(fitsio.read('/global/cfs/cdirs/desi/users/rongpu/data/lrg_xcorr/catalogs/dr9_lrg_1.1.1_pzbins_20221204.fits'))
print(len(cat))

12338990


In [13]:
min_nobs = 2
maskbits = [1, 8, 9, 11, 12, 13]

mask = (cat['PIXEL_NOBS_G']>=min_nobs) & (cat['PIXEL_NOBS_R']>=min_nobs) & (cat['PIXEL_NOBS_Z']>=min_nobs)
print(np.sum(~mask)/len(mask))

mask_clean = np.ones(len(cat), dtype=bool)
for bit in maskbits:
    mask_clean &= (cat['MASKBITS'] & 2**bit)==0
print(np.sum(~mask_clean)/len(mask_clean))
mask &= mask_clean

cat = cat[mask]
print(len(cat))

0.05278722164455924
0.10460434768161737
10523888


In [14]:
for photsys in ['N', 'S']:
    print(photsys)
    for bin_index in range(1, 5):
        mask = cat['PHOTSYS']==photsys
        mask &= cat['pz_bin']==bin_index
        print('bin', bin_index, np.sum(mask))
    print()

N
bin 1 381322
bin 2 681670
bin 3 746325
bin 4 678831

S
bin 1 1080778
bin 2 1927349
bin 3 2102918
bin 4 1886210



In [17]:
counts_path = '/global/cfs/cdirs/desi/users/rongpu/lrg_xcorr/magnification/counts/lrg_counts.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)

In [18]:
for photsys in ['N', 'S']:
    if photsys=='N':
        field = 'north'
    else:
        field = 'south'
    print(photsys)
    mask = cat['PHOTSYS']==photsys
    print('all', np.sum(mask), counts['{}_all_1.000'.format(field)], 'diff: {:.3f}%'.format((counts['{}_all_1.000'.format(field)]-np.sum(mask))/np.sum(mask)*100))
    for bin_index in range(1, 5):
        mask = cat['PHOTSYS']==photsys
        mask &= cat['pz_bin']==bin_index
        print('bin', bin_index, np.sum(mask), counts['{}_bin_{}_1.000'.format(field, bin_index)], 'diff: {:.3f}%'.format((counts['{}_bin_{}_1.000'.format(field, bin_index)]-np.sum(mask))/np.sum(mask)*100))
    print()

N
all 2747211 2747213 diff: 0.000%
bin 1 381322 381392 diff: 0.018%
bin 2 681670 681533 diff: -0.020%
bin 3 746325 746356 diff: 0.004%
bin 4 678831 678762 diff: -0.010%

S
all 7776677 7776682 diff: 0.000%
bin 1 1080778 1080746 diff: -0.003%
bin 2 1927349 1927588 diff: 0.012%
bin 3 2102918 2102651 diff: -0.013%
bin 4 1886210 1886317 diff: 0.006%



--------
# Number count slope

In [5]:
counts_path = '/global/cfs/cdirs/desi/users/rongpu/lrg_xcorr/magnification/counts/lrg_counts.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)
    
magnify_low, magnify_high = 0.99, 1.01
dmag = (-2.5 * np.log10(magnify_low)) - (-2.5 * np.log10(magnify_high))

for field in ['north', 'south', 'combined']:
    print(field)
    if field!='combined':
        count_low = counts['{}_all_{:.3f}'.format(field, magnify_low)]
        count_high = counts['{}_all_{:.3f}'.format(field, magnify_high)]
        count_center = counts['{}_all_{:.3f}'.format(field, 1.)]
    else:
        count_low = counts['north_all_{:.3f}'.format(magnify_low)] + counts['south_all_{:.3f}'.format(magnify_low)]
        count_high = counts['north_all_{:.3f}'.format(magnify_high)] + counts['south_all_{:.3f}'.format(magnify_high)]
        count_center = counts['north_all_{:.3f}'.format(1.)] + counts['south_all_{:.3f}'.format(1.)]
    s = (np.log10(count_high)-np.log10(count_low)) / dmag
    s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
    print('all:  s = {:.3f} +- {:.3f}'.format(s, s_err))
    for bin_index in range(1, 5):
        if field!='combined':
            count_low = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_low)]
            count_high = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_high)]
            count_center = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, 1.)]
        else:
            count_low = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_low)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_low)]
            count_high = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_high)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_high)]
            count_center = counts['north_bin_{}_{:.3f}'.format(bin_index, 1.)] + counts['south_bin_{}_{:.3f}'.format(bin_index, 1.)]
        s = (np.log10(count_high)-np.log10(count_low)) / dmag
        s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
        print('bin {}:  s = {:.3f} +- {:.3f}'.format(bin_index, s, s_err))
        # Sanity check
        # print('bin {}   {}  {} diff:{:.4f}%'.format(bin_index, (count_low+count_high)/2, count_center, 100*((count_low+count_high)/2/count_center-1)))
    print()

north
all:  s = 1.021 +- 0.012
bin 1:  s = 0.978 +- 0.032
bin 2:  s = 1.070 +- 0.024
bin 3:  s = 0.984 +- 0.023
bin 4:  s = 0.981 +- 0.024

south
all:  s = 1.002 +- 0.007
bin 1:  s = 0.932 +- 0.019
bin 2:  s = 1.045 +- 0.014
bin 3:  s = 1.001 +- 0.014
bin 4:  s = 0.948 +- 0.015

combined
all:  s = 1.007 +- 0.006
bin 1:  s = 0.944 +- 0.017
bin 2:  s = 1.051 +- 0.012
bin 3:  s = 0.996 +- 0.012
bin 4:  s = 0.957 +- 0.012



No fiberflux factor:

In [8]:
counts_path = '/global/cfs/cdirs/desi/users/rongpu/lrg_xcorr/magnification/counts/lrg_counts_no_ff.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)
    
magnify_low, magnify_high = 0.99, 1.01
dmag = (-2.5 * np.log10(magnify_low)) - (-2.5 * np.log10(magnify_high))

for field in ['north', 'south', 'combined']:
    print(field)
    if field!='combined':
        count_low = counts['{}_all_{:.3f}'.format(field, magnify_low)]
        count_high = counts['{}_all_{:.3f}'.format(field, magnify_high)]
        count_center = counts['{}_all_{:.3f}'.format(field, 1.)]
    else:
        count_low = counts['north_all_{:.3f}'.format(magnify_low)] + counts['south_all_{:.3f}'.format(magnify_low)]
        count_high = counts['north_all_{:.3f}'.format(magnify_high)] + counts['south_all_{:.3f}'.format(magnify_high)]
        count_center = counts['north_all_{:.3f}'.format(1.)] + counts['south_all_{:.3f}'.format(1.)]
    s = (np.log10(count_high)-np.log10(count_low)) / dmag
    s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
    print('all:  s = {:.3f} +- {:.3f}'.format(s, s_err))
    for bin_index in range(1, 5):
        if field!='combined':
            count_low = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_low)]
            count_high = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_high)]
            count_center = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, 1.)]
        else:
            count_low = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_low)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_low)]
            count_high = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_high)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_high)]
            count_center = counts['north_bin_{}_{:.3f}'.format(bin_index, 1.)] + counts['south_bin_{}_{:.3f}'.format(bin_index, 1.)]
        s = (np.log10(count_high)-np.log10(count_low)) / dmag
        s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
        print('bin {}:  s = {:.3f} +- {:.3f}'.format(bin_index, s, s_err))
        # Sanity check
        # print('bin {}   {}  {} diff:{:.4f}%'.format(bin_index, (count_low+count_high)/2, count_center, 100*((count_low+count_high)/2/count_center-1)))
    print()

north
all:  s = 1.140 +- 0.012
bin 1:  s = 0.984 +- 0.032
bin 2:  s = 1.095 +- 0.024
bin 3:  s = 1.092 +- 0.023
bin 4:  s = 1.253 +- 0.024

south
all:  s = 1.134 +- 0.007
bin 1:  s = 0.940 +- 0.019
bin 2:  s = 1.077 +- 0.014
bin 3:  s = 1.122 +- 0.014
bin 4:  s = 1.256 +- 0.015

combined
all:  s = 1.135 +- 0.006
bin 1:  s = 0.952 +- 0.017
bin 2:  s = 1.082 +- 0.012
bin 3:  s = 1.114 +- 0.012
bin 4:  s = 1.255 +- 0.012



No photo-z shift:

In [9]:
counts_path = '/global/cfs/cdirs/desi/users/rongpu/lrg_xcorr/magnification/counts//lrg_counts_no_pz_mag.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)
    
magnify_low, magnify_high = 0.99, 1.01
dmag = (-2.5 * np.log10(magnify_low)) - (-2.5 * np.log10(magnify_high))

for field in ['north', 'south', 'combined']:
    print(field)
    if field!='combined':
        count_low = counts['{}_all_{:.3f}'.format(field, magnify_low)]
        count_high = counts['{}_all_{:.3f}'.format(field, magnify_high)]
        count_center = counts['{}_all_{:.3f}'.format(field, 1.)]
    else:
        count_low = counts['north_all_{:.3f}'.format(magnify_low)] + counts['south_all_{:.3f}'.format(magnify_low)]
        count_high = counts['north_all_{:.3f}'.format(magnify_high)] + counts['south_all_{:.3f}'.format(magnify_high)]
        count_center = counts['north_all_{:.3f}'.format(1.)] + counts['south_all_{:.3f}'.format(1.)]
    s = (np.log10(count_high)-np.log10(count_low)) / dmag
    s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
    print('all:  s = {:.3f} +- {:.3f}'.format(s, s_err))
    for bin_index in range(1, 5):
        if field!='combined':
            count_low = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_low)]
            count_high = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_high)]
            count_center = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, 1.)]
        else:
            count_low = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_low)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_low)]
            count_high = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_high)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_high)]
            count_center = counts['north_bin_{}_{:.3f}'.format(bin_index, 1.)] + counts['south_bin_{}_{:.3f}'.format(bin_index, 1.)]
        s = (np.log10(count_high)-np.log10(count_low)) / dmag
        s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
        print('bin {}:  s = {:.3f} +- {:.3f}'.format(bin_index, s, s_err))
        # Sanity check
        # print('bin {}   {}  {} diff:{:.4f}%'.format(bin_index, (count_low+count_high)/2, count_center, 100*((count_low+count_high)/2/count_center-1)))
    print()

north
all:  s = 1.021 +- 0.012
bin 1:  s = 1.025 +- 0.032
bin 2:  s = 1.002 +- 0.024
bin 3:  s = 0.877 +- 0.023
bin 4:  s = 1.161 +- 0.024

south
all:  s = 1.002 +- 0.007
bin 1:  s = 0.980 +- 0.019
bin 2:  s = 1.005 +- 0.014
bin 3:  s = 0.860 +- 0.014
bin 4:  s = 1.151 +- 0.015

combined
all:  s = 1.007 +- 0.006
bin 1:  s = 0.991 +- 0.017
bin 2:  s = 1.005 +- 0.012
bin 3:  s = 0.864 +- 0.012
bin 4:  s = 1.154 +- 0.012

